# 📓 Lab 01: Production Hybrid RAG & Re-ranking Cookbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TontonYuta/Applied-AI-Engineering-Handbook/blob/main/notebooks/01_interactive_rag_cookbook.ipynb)

> **Mục tiêu thực hành trực tiếp trên trình duyệt:**
> 1. Phân đoạn văn bản bằng thuật toán Recursive Character Chunker.
> 2. Chạy song song Sparse BM25 và Dense Vector Search.
> 3. Dung hợp thứ hạng bằng công thức Reciprocal Rank Fusion (RRF $k=60$).

In [ ]:
# Cài đặt các thư viện cơ bản nếu chạy trên Google Colab
!pip install -q numpy pydantic rich

In [ ]:
import math
import re
import hashlib
import numpy as np
from typing import List, Dict, Tuple

print("✅ Môi trường đã sẵn sàng!")

## Bước 1: Khởi tạo Thuật toán BM25 Okapi cho từ khóa chính xác

In [ ]:
class BM25Okapi:
    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.corpus_size = 0
        self.avgdl = 0.0
        self.doc_freqs = []
        self.idf = {}
        self.doc_len = []
        self.doc_ids = []
        self.corpus_texts = []

    @staticmethod
    def tokenize(text: str) -> List[str]:
        return re.findall(r"[\w\-]+", text.lower())

    def fit(self, doc_ids: List[str], texts: List[str]):
        self.doc_ids = doc_ids
        self.corpus_texts = texts
        self.corpus_size = len(texts)
        self.doc_len = []
        self.doc_freqs = []
        df = {}
        total_len = 0

        for text in texts:
            tokens = self.tokenize(text)
            length = len(tokens)
            self.doc_len.append(length)
            total_len += length
            freqs = {}
            for t in tokens:
                freqs[t] = freqs.get(t, 0) + 1
            self.doc_freqs.append(freqs)
            for t in freqs.keys():
                df[t] = df.get(t, 0) + 1

        self.avgdl = total_len / self.corpus_size if self.corpus_size > 0 else 0.0
        for t, freq in df.items():
            self.idf[t] = math.log((self.corpus_size - freq + 0.5) / (freq + 0.5) + 1.0)

    def search(self, query: str, top_k: int = 3) -> List[Tuple[str, float]]:
        tokens = self.tokenize(query)
        scores = []
        for idx in range(self.corpus_size):
            score = 0.0
            doc_len = self.doc_len[idx]
            freqs = self.doc_freqs[idx]
            for t in tokens:
                if t not in freqs: continue
                tf = freqs[t]
                idf = self.idf.get(t, 0.0)
                denom = tf + self.k1 * (1.0 - self.b + self.b * (doc_len / self.avgdl))
                score += idf * (tf * (self.k1 + 1.0)) / denom
            scores.append((self.doc_ids[idx], round(score, 4)))
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

print("✔ Đã biên dịch BM25Okapi!")

## Bước 2: Thử nghiệm tìm kiếm chính xác mã lỗi hệ thống

In [ ]:
docs = [
    "Mã lỗi ERR-503-TIMEOUT xảy ra khi cụm máy chủ Inference bị quá tải VRAM.",
    "Quy định bảo hành và đổi trả sản phẩm trong vòng 30 ngày.",
    "Hướng dẫn kết nối cơ sở dữ liệu qua giao thức gRPC an toàn."
]
doc_ids = ["chunk_01", "chunk_02", "chunk_03"]

bm25 = BM25Okapi()
bm25.fit(doc_ids, docs)

results = bm25.search("ERR-503-TIMEOUT quá tải")
for doc_id, score in results:
    print(f"ID: {doc_id} | Điểm BM25: {score}")